First we are goin to load the data, then split the data.

In [1]:
def extract_L_set(filepath):
    l_set = []
    capture = False
    with open(filepath, 'r') as file:
        for line in file:
            line = line.strip()
        
            # Start capturing when we see the header
            if line.startswith("L set:"):
                capture = True
                continue
        
            # Stop capturing when we reach the B vector
            if line.startswith("B vector:"):
                capture = False
                break
            
            # If we are in the capture zone and the line looks like a list
            if capture and line.startswith("["):
                # ast.literal_eval safely converts the string "[1, 2...]" into a Python list
                row = ast.literal_eval(line)
                l_set.append(row)
    return l_set

In [2]:
import numpy as np
import ast #abstract syntax tree module for safe evaluation of string.
loaded_train_test_data = []
#first we loading the L set from 2003 to 2023
filePath = '../ED_Calculation/2003_2023/results/finalize_L_set_from_2003_to_2023.txt'
loaded_train_test_data = np.array(extract_L_set(filePath))


In [3]:
from sklearn.model_selection import train_test_split
#separate the test_data for Feature and Target
X = loaded_train_test_data[:, :5] #first five col of the row is feature
y = loaded_train_test_data[:, 5] # 6th value is the target

#split for train and val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train {X_train.shape}")
print(f"Shape of y_train{y_train.shape}")
print(f"Shape of X_val{X_val.shape}")
print(f"Shape of y_val{y_val.shape}")

Shape of X_train (116, 5)
Shape of y_train(116,)
Shape of X_val(29, 5)
Shape of y_val(29,)


It is time to define the test data from **2024 to 2025**.

In [4]:
loaded_test_data = []
test_filepath = "../ED_Calculation/2024_current/results/final_result_df_2024_2025.txt"
loaded_test_data = np.array(extract_L_set(test_filepath))
X_test = loaded_test_data[:, :5]
y_test = loaded_test_data[:, 5]


Now we are going to make a Feed Forward Neural Network, which has input layer with 5 feature, a hidden layer with initial neuron size 10 with signmoid function, and a signle neuron output layer with linear function.

In [10]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input


#Create a model
ffnn = Sequential()
# Input layer (5 features) tells the model to expect 5 columns
ffnn.add(Input(shape=(X_train.shape[1],)))
# Hidden layer (50 neurons) 
ffnn.add(Dense(units=50, activation="sigmoid",))
ffnn.add(Dense(units=1, activation="linear"))
ffnn.compile(optimizer="adam", loss='mse', metrics=['mae'])
ffnn.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 50)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 351 (1.37 KB)

 Trainable params: 351 (1.37 KB)

 Non-trainable params: 0 (0.00 B)

Now run and train the model

In [6]:

ffnn_history = ffnn.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), verbose=1,)
print("Model compiled and trained successfully.")

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 109ms/step - loss: 1.3727 - mae: 1.0424 - val_loss: 1.6658 - val_mae: 1.1733
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 1.1374 - mae: 0.9344 - val_loss: 1.3979 - val_mae: 1.0527
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.9388 - mae: 0.8351 - val_loss: 1.1650 - val_mae: 0.9352
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.7553 - mae: 0.7420 - val_loss: 0.9710 - val_mae: 0.8245
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.6308 - mae: 0.6736 - val_loss: 0.8062 - val_mae: 0.7237
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.5208 - mae: 0.6044 - val_loss: 0.6746 - val_mae: 0.6528
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.4429 - mae: 0.5453 - val_loss: 0.5719 - val_mae: 0.5962
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.3894 - mae: 0.5042 - val_loss: 0.4948 - val_mae: 0.5527
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.3522 - mae: 0.4759 -

In [7]:
test_results = ffnn.evaluate(X_test, y_test, verbose=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0683 - mae: 0.2132


In [8]:
print(f"Test MAE: {test_results[1]:.4}")


Test MAE: 0.2132


In [9]:
import pandas as pd
predictions = ffnn.predict(X_test)

# Create a comparison table
results_df = pd.DataFrame({
    'Actual Value': y_test,
    'Predicted Value': predictions.flatten(),
    'Difference': y_test - predictions.flatten()
})

print("Predictions for the 2024-2025 Test Set:")
display(results_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Predictions for the 2024-2025 Test Set:


,Actual Value,Predicted Value,Difference
0,-0.017473,0.016395,-0.033868
1,-0.339443,0.051059,-0.390503
2,0.031541,0.033494,-0.001953
3,-0.215161,0.070775,-0.285936
4,-0.087107,0.044237,-0.131344
5,0.485383,0.051514,0.433869
6,-0.220254,0.178562,-0.398817
7,0.307394,0.020643,0.286752
8,0.306878,0.074292,0.232586
9,0.113420,0.083893,0.029527
